In [1]:
%pip freeze | grep sagemaker

sagemaker==2.235.2
sagemaker-core==1.0.77
sagemaker-experiments==0.1.45
sagemaker_training==4.9.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker import get_execution_role

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [ ]:
role = get_execution_role()
sess = sagemaker.Session()

# ============================================
# S3 설정 변수
# ============================================
S3_BUCKET = "retail-mlops-edu-2026"
S3_USER = "kunops"
S3_INPUT_PREFIX = f"edu-2w/{S3_USER}/input"
S3_OUTPUT_PREFIX = f"edu-2w/{S3_USER}/output"
S3_CODE_PREFIX = f"edu-2w/{S3_USER}/code"

NOTEBOOK_FILE = "train_titanic_lightgbm.ipynb"

## Model Train job 실행

In [ ]:
# ============================================
# Estimator 생성
# ============================================
estimator = Estimator(
    image_uri="155954279556.dkr.ecr.us-east-1.amazonaws.com/gs-automl-base-containers/tabular-kunops-311_sm:1.0",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",

    # 하이퍼파라미터 → run_pm.py parse_args() → papermill → 노트북 parameters 셀
    hyperparameters={
        # S3 설정 (run_pm.py에서 S3 I/O에 사용)
        "s3_bucket": S3_BUCKET,
        "s3_user": S3_USER,
        "s3_notebook": notebook_s3_path,
        "s3_input_prefix": S3_INPUT_PREFIX,
        "s3_output_prefix": S3_OUTPUT_PREFIX,
        # LightGBM hyperparameters
        "objective": "binary",
        "metric": "binary_logloss",
        "num_leaves": "31",
        "learning_rate": "0.1",
        "n_estimators": "100",
        "max_depth": "10",
        "random_state": "42",
        "verbose": "0",
        # train/val split
        "val_ratio": "0.2",
        "random_state_split": "42",
    },

    base_job_name="train-titanic-lightgbm",
    sagemaker_session=sess,

    tags=[
        {"Key": "Environment", "Value": "dev"},
        {"Key": "Project", "Value": "automl"},
        {"Key": "Owner", "Value": S3_USER},
        {"Key": "CostCenter", "Value": "gs-retail"},
    ],

    output_path=f"s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}",
)

In [ ]:
# ============================================
# 노트북 + 학습 데이터를 S3에 업로드
# ============================================
# 1) 실행할 노트북 업로드
notebook_s3_path = sess.upload_data(
    path=NOTEBOOK_FILE,
    bucket=S3_BUCKET,
    key_prefix=S3_CODE_PREFIX
)
print(f"Notebook uploaded to: {notebook_s3_path}")

# 2) 학습 데이터 업로드
train_s3_path = sess.upload_data(
    path='train.csv',
    bucket=S3_BUCKET,
    key_prefix=S3_INPUT_PREFIX
)
print(f"Training data uploaded to: {train_s3_path}")

In [6]:
# ============================================
# Training Job 실행
# ============================================
try:
    estimator.fit({
        'training': train_s3_path
    })

    print("=" * 60)
    print("✅ Training Job 완료!")
    print(f"   Job Name: {estimator.latest_training_job.name}")
    print(f"   Model Artifact: {estimator.model_data}")
    print("=" * 60)

except Exception as e:
    print(f"❌ Training Job 실패: {e}")

INFO:sagemaker:Creating training-job with name: train-titanic-lightgbm-2026-02-27-16-36-49-749


2026-02-27 16:36:52 Starting - Starting the training job...
2026-02-27 16:37:07 Starting - Preparing the instances for training...
2026-02-27 16:37:54 Downloading - Downloading the training image......
2026-02-27 16:38:45 Training - Training image download completed. Training in progress..2026-02-27 16:38:49,808 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-02-27 16:38:49,808 sagemaker-training-toolkit INFO     Failed to parse hyperparameter metric value binary_logloss to Json.
Returning the value itself
2026-02-27 16:38:49,808 sagemaker-training-toolkit INFO     Failed to parse hyperparameter objective value binary to Json.
Returning the value itself
2026-02-27 16:38:49,809 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-02-27 16:38:49,824 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-02-27 16:38:49,825 sagemaker-training-toolkit INFO     Failed to parse hy

## 결과 확인

In [ ]:
# ============================================================
# Step 3: 결과 노트북 다운로드 및 확인
# ============================================================

import boto3
import os
import pandas as pd

s3 = boto3.client('s3')

# ============================================
# 실행된 노트북 다운로드
# ============================================
output_nb_key = f'{S3_OUTPUT_PREFIX}/train_titanic_lightgbm_output.ipynb'
local_path = 'titanic_output.ipynb'

s3.download_file(S3_BUCKET, output_nb_key, local_path)

print(f"✅ 결과 노트북 다운로드 완료: {local_path}")
print(f"   Jupyter에서 열어서 확인하세요!")

# ============================================
# 모델 다운로드
# ============================================
job_name = estimator.latest_training_job.name
print(f"Job Name: {job_name}")

model_key = f"edu-2w/{S3_USER}/model/model.tar.gz"
model_local = 'model.tar.gz'
s3.download_file(S3_BUCKET, model_key, model_local)

import tarfile
with tarfile.open(model_local, 'r:gz') as tar:
    tar.extractall('model')

print(f"✅ 모델 다운로드 완료: ./model/")

# ============================================
# 모델 로드 및 테스트
# ============================================
import joblib

model = joblib.load('model/titanic_model.joblib')

val_key = f"edu-2w/{S3_USER}/data/val/validation.csv"
val_local = "test.csv"
s3.download_file(S3_BUCKET, val_key, val_local)
test_df = pd.read_csv(val_local)
target_col = "survived" if "survived" in test_df.columns else "target"
X_test = test_df.drop(target_col, axis=1)
y_test = test_df[target_col]
predictions = model.predict(X_test)

print(f"✅ 예측 완료: {len(predictions)} samples")